# NextBrain labels on the Allen ontology tree

This notebook builds a **label tree** for the image generator. It has the same format as
`data/inputs/nextbrain_tree_v1.json`, but its structure comes from the **Allen developmental
ontology** instead of the old hand-made NextBrain hierarchy.

**What the tree looks like**

```
root                                        level 0
├── background            (label 0)         level 1, leaf, intensity always 0
├── FGM gray matter of forebrain            level 1  ← a "color"
│   └── Tel telencephalon                   level 2  ← Allen nodes ...
│       └── ...
│           └── putamen   (label 79)        leaf     ← a NextBrain label
├── FWM white matter of forebrain           level 1
│   └── ...
└── ...
```

- **Leaves** = the NextBrain labels listed in `NextBrainLabels.yaml` (one leaf per label name).
- **Internal nodes** = Allen ontology nodes. Only the ones that have at least one leaf below them
  are kept.
- **Level 1** = the "colors" of the generator: the Allen tissue classes (gray matter, white
  matter, ventricles, surface structures, blood vessels) of forebrain, midbrain and hindbrain.
  Each color gets its own independent base intensity in the generator.
- **Background** (label 0) is a leaf directly under the root, with intensity 0.
- **Hand-made decisions** are all in two dicts of step 4.3: `MANUAL` (leaf → its Allen node,
  when the name alone does not find it) and `PLACE_UNDER` (leaf → its father, for leaves without
  a usable Allen node, e.g. `midbrain_(mesencephalon)` under MGM). With them, **every NextBrain
  label is in the tree**.
- Every node also stores its Allen information: `allen_id`, `allen_acronym`, `allen_name`,
  `allen_color`.

**Steps**

1. Load the inputs (NextBrain leaves and Allen ontology).
2. Make the Allen ontology easy to walk (a flat table).
3. Choose the 15 colors (level 1).
4. Find, for every NextBrain leaf, its Allen node (or, for a few, its father, chosen by hand).
   Leaves not found → **list of unmatched leaves** (empty now).
5. Decide which Allen nodes stay in the tree.
6. Build the tree with `AllenLabelTree`.
7. List the Allen nodes that were removed because they have no leaves (**fathers without leaves**).
8. Check the tree.
9. Look at it and save it.

## 1. Load the inputs

We use `nextbrain_utils` (the `nextbrain-utils` package of this folder) to read both inputs:

- `io.load_yaml(LUTDIR / "NextBrainLabels.yaml")` → the NextBrain leaves;
- `io.load_ontology()` → the Allen developmental ontology (`AllenBrainOntologyDev.json`).

We also import `AllenLabelTree` from `allen_tree.py` (next to this notebook). It is the tree
class used by the generator (`LabelTree`), extended to accept the `allen_*` fields; see step 6.

In [1]:
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

from nextbrain_utils import io
from nextbrain_utils import to_allen as TA
from allen_tree import AllenLabelTree

LUTDIR = Path(io.__file__).parent / "lut"                            # nextbrain-utils lookup tables
OUT_TREE = Path("data/outputs/nextbrain_tree_allen_gyral.json")   # the only output file

### 1.1 NextBrain leaves

`NextBrainLabels.yaml` maps a **NextBrain name** to its **label ID** (the value painted in the
segmentation). There is one special case: `white_matter_of_hindbrain` maps to a *list* of two
labels (846 cerebellum, 611 brainstem). It is still one leaf; the tree format allows a node to
paint several labels (`labels` is a list).

The entry `Unknown: 0` is the background.

In [2]:
leaves = io.load_yaml(LUTDIR / "NextBrainLabels.yaml")

print(len(leaves), "entries. The first ones:")
for name in list(leaves)[:6]:
    print(f"   {name}: {leaves[name]}")
print("Entries with several labels:", {k: v for k, v in leaves.items() if isinstance(v, list)})

334 entries. The first ones:
   Unknown: 0
   white_matter_of_forebrain: 7
   head_of_caudate: 48
   anterior_horn_of_lateral_ventricle: 50
   septum_pellucidum: 68
   putamen: 79
Entries with several labels: {'white_matter_of_hindbrain': [846, 611]}


### 1.2 Allen ontology

The ontology is a nested dictionary. Each node has an `id`, a `name`, an `acronym`, a color
(`color_hex_triplet`) and a list of `children`. Its first levels look like this
(`neural plate → neural tube → brain → forebrain / midbrain / hindbrain → tissue classes`):

In [3]:
ont = io.load_ontology()

def print_allen(node, depth=0, max_depth=4):
    print("    " * depth + f"{node['acronym']:4s} {node['id']:>9}  {node['name']}")
    if depth < max_depth:
        for child in node.get("children", []):
            print_allen(child, depth + 1, max_depth)

print_allen(ont)

NP       10153  neural plate
    NT       10154  neural tube
        Br       10155  brain
            F        10156  forebrain (prosencephalon)
                FGM      10157  gray matter of forebrain
                FWM      10557  white matter of forebrain
                FV       10595  ventricles of forebrain
                FSS      10609  surface structures of forebrain
                fbv  266441685  blood vessels of forebrain
            M        10648  midbrain (mesencephalon)
                MGM      10649  gray matter of midbrain
                MWM      10650  white matter of midbrain
                MV       10651  ventricle of midbrain
                MSS      10652  surface structures of midbrain
                mbv  266441705  blood vessels of midbrain
            H        10653  hindbrain (rhombencephalon)
                HGM      10654  gray matter of the hindbrain
                HWM      10668  white matter of hindbrain
                HV       10669  ventricles o

## 2. A flat table of the Allen ontology

A nested dictionary is hard to walk *upwards* (from a node to its parent). So we store every node
in a flat dictionary `allen`, keyed by Allen id, with its parent id and its children ids.

`ancestors(aid)` returns all the Allen ids above `aid`, from its parent up to the root.

In [4]:
allen = {}   # allen id -> {"name", "acronym", "color", "parent", "children"}

def add_to_table(node, parent=None):
    allen[node["id"]] = {
        "name": node["name"],
        "acronym": node["acronym"],
        "color": node["color_hex_triplet"],
        "parent": parent,
        "children": [child["id"] for child in node.get("children", [])],
    }
    for child in node.get("children", []):
        add_to_table(child, node["id"])

add_to_table(ont)

def ancestors(aid):
    out = []
    while allen[aid]["parent"] is not None:
        aid = allen[aid]["parent"]
        out.append(aid)
    return out

print(len(allen), "Allen nodes")
print("Example, putamen:", allen[10338])
print("Its ancestors:", [allen[a]["acronym"] for a in ancestors(10338)])

3317 Allen nodes
Example, putamen: {'name': 'putamen', 'acronym': 'Pu', 'color': 'A8B485', 'parent': 10333, 'children': [11537, 11538, 146034754, 11542]}
Its ancestors: ['STR', 'BN', 'CN', 'Tel', 'FGM', 'F', 'Br', 'NT', 'NP']


## 3. The colors (level 1)

The level-1 nodes are the 15 Allen tissue classes below forebrain (F), midbrain (M) and
hindbrain (H):

| | gray matter | white matter | ventricles | surface structures | blood vessels |
|---|---|---|---|---|---|
| forebrain | FGM | FWM | FV | FSS | fbv |
| midbrain  | MGM | MWM | MV | MSS | mbv |
| hindbrain | HGM | HWM | HV | HSS | hbv |

Everything above them (neural plate, neural tube, brain, F/M/H) is not used: the root of our
tree replaces it.

`color_of(aid)` tells which color an Allen node belongs to (or `None` if it is not below any).

In [5]:
COLORS = ["FGM", "FWM", "FV", "FSS", "fbv",
          "MGM", "MWM", "MV", "MSS", "mbv",
          "HGM", "HWM", "HV", "HSS", "hbv"]

acronym_to_id = {a["acronym"]: aid for aid, a in allen.items()}
color_ids = [acronym_to_id[acronym] for acronym in COLORS]

def color_of(aid):
    for a in [aid] + ancestors(aid):
        if a in color_ids:
            return a
    return None

pd.DataFrame({"acronym": COLORS, "allen_id": color_ids,
              "name": [allen[c]["name"] for c in color_ids]})

,acronym,allen_id,name
0,FGM,10157,gray matter of forebrain
1,FWM,10557,white matter of forebrain
2,FV,10595,ventricles of forebrain
3,FSS,10609,surface structures of forebrain
4,fbv,266441685,blood vessels of forebrain
5,MGM,10649,gray matter of midbrain
6,MWM,10650,white matter of midbrain
7,MV,10651,ventricle of midbrain
8,MSS,10652,surface structures of midbrain
9,mbv,266441705,blood vessels of midbrain


## 4. Find the Allen node of every NextBrain leaf

### 4.1 The background is set aside

`Unknown` (label 0) is the background, not a brain structure, so we do not look for it in the
ontology. It will be added directly under the root in step 6.

We also turn every label into a list (`79` → `[79]`), so all leaves look the same.

In [6]:
BACKGROUND = "Unknown"

labels_of = {name: (lab if isinstance(lab, list) else [lab]) for name, lab in leaves.items()}
background_labels = labels_of.pop(BACKGROUND)

print("background labels:", background_labels)
print(len(labels_of), "leaves to match")

background labels: [0]
333 leaves to match


### 4.2 Compare names, not spellings

NextBrain and Allen write names differently (`substantia_nigra__compact_part` vs.
`substantia nigra, compact part`). `TA.normalize_name` from nextbrain-utils removes those
differences: lowercase, no commas, `-` and `_` become spaces, repeated spaces collapse, and the
`ctx-` / `rh-` prefixes are removed.

We build `name_index`: normalised Allen name → list of Allen ids with that name. Usually the
list has one id, but a few names exist twice in the ontology.

In [7]:
for s in ["substantia_nigra__compact_part", "substantia nigra, compact part", "ctx-rh-superiorfrontal"]:
    print(f"{s!r:35s} -> {TA.normalize_name(s)!r}")

name_index = defaultdict(list)
for aid, a in allen.items():
    name_index[TA.normalize_name(a["name"])].append(aid)

'substantia_nigra__compact_part'    -> 'substantia nigra compact part'
'substantia nigra, compact part'    -> 'substantia nigra compact part'
'ctx-rh-superiorfrontal'            -> 'superiorfrontal'


### 4.3 Special cases and hand-made decisions

Most leaves (about 290) are found by their name alone. The others need one of these:

1. **Cortex.** NextBrain's cortex labels (`ctx-rh-*`, 2001–2035) are the FreeSurfer
   Desikan-Killiany parcels. Allen has no node with these names. nextbrain-utils has a
   hand-made table that gives the Allen name for each parcel. We use the **gyral** one
   (`TA._fscortex_to_allen_gyral()`), which places the parcels on Allen's gyri, under FSS
   (e.g. `superiorfrontal` → `superior frontal gyrus`).

2. **`MANUAL`: the Allen node is given by hand.** The leaf *is* an Allen structure, but its name
   does not find it (one name, two Allen nodes; or a different spelling). The leaf is then
   treated exactly like a name match (rules of step 5).

   | Leaf | Allen node | Why |
   |---|---|---|
   | `lateral_nucleus` (377) | 10367 `La`, lateral nucleus of the amygdala | the name exists twice in Allen; the other one (12410) is a pontine nucleus |
   | `dorsal_raphe_nucleus` (521) | 12223 `DR`, midbrain | the name exists twice; the other one (12472) is the pontine part |
   | `decussation_of_trochlear_nerve_fibres` (842) | 12338 `x4`, decussation of trochlear nerve fibers (`M → MWM → x4`) | same structure; Allen spells it "fibers" and not "fibres" |
   | `reticular_formation_/_cuneiform_area` (844) | 12239 `MRF`, midbrain reticular formation (`M → MGM → MTg → MRF`) | the label covers the midbrain reticular formation around the cuneiform nucleus; in Allen the cuneiform nucleus (`CnF`) is a part of `MRF` |

   842 and 844 are midbrain structures in Allen (and in the old NextBrain table:
   `L4_subregion = Midbrain`), so they stay in the midbrain: 842 in MWM, 844 in MGM.

3. **Nodes outside the colors.** If the Allen node found is not below one of the 15 colors,
   the leaf has no color to hang from. This happens to `midbrain_(mesencephalon)`, solved
   with `PLACE_UNDER` (case 4).

4. **`PLACE_UNDER`: the father is given by hand.** For leaves with no Allen node of their own,
   or whose Allen node cannot be used, we choose the Allen node they hang from. The leaf is
   always a direct **child** of that father. Its `allen_*` fields keep its own Allen node if it
   has one (384 → `M`) and are left empty if it has none: its *position* is a choice, but its
   *identity* is not invented. The leaf's `description` records the decision.

   | Leaf | Father | Own Allen node | Why |
   |---|---|---|---|
   | `midbrain_(mesencephalon)` (384) | 10649 `MGM`, gray matter of midbrain | 10648 `M` (above the colors) | see below |
   | `superficial_pulvinar_nucleus` (811) | 10409 `Pul`, pulvinar of thalamus | none | a NextBrain subdivision of the pulvinar; Allen has no node for it |
   | `diffuse_pulvinar_nucleus` (813) | 10409 `Pul` | none | same |
   | `parietotemporopontine_tract` (827) | 10668 `HWM`, white matter of hindbrain | none | corticopontine fibres from parietal/temporal cortex; Allen only has corticopontine fibres as a whole (its pontine part `cpn` is under HWM) |
   | `corticonuclear_tract` (828) | 10668 `HWM` | none | corticonuclear = corticobulbar; Allen's hindbrain corticobulbar tract (`cbu-h`) is under HWM |
   | `frontopontine_tract` (829) | 10668 `HWM` | none | corticopontine fibres from frontal cortex; same as 827 |

   **Why `midbrain_(mesencephalon)` goes under MGM.** Its name matches Allen **`M` midbrain
   (mesencephalon)**, the whole midbrain division, which is *above* the colors
   (`Br → M → MGM / MWM / MV / MSS / mbv`). We cannot leave it out of the tree (it is real brain
   tissue in every segmentation), so:

   - *What the label is.* In NextBrain, 384 is the midbrain **"whole level (unassigned)"**
     (description in `nextbrain_tree_v1.json`): the midbrain voxels not given to any nucleus or
     tract. That is mostly tegmentum and reticular gray substance.
   - *Its white matter is already elsewhere.* The midbrain white matter has its own leaf,
     `white_matter_of_midbrain` (431), under MWM. So what is left in 384 is mostly not white
     matter.
   - *Same as the pons.* `pons` (580) is "pons, unassigned" too, and in Allen it is gray matter
     (`HGM → Met → Pn`). Putting the midbrain under MGM treats both brainstem divisions alike.
   - *Same rule as the other "unassigned" leaves.* `thalamus` under `THM`, or
     `white_matter_of_forebrain` under `FWM`: an "X, unassigned" leaf is a child of the node
     that contains X's parts (step 5).
   - *In the generator.* Its mean is the MGM mean plus the leaf jitter, and at coarse
     resolution it falls back to MGM together with the rest of the midbrain gray matter.

In [8]:
DK_TO_ALLEN = TA._fscortex_to_allen_gyral()     # "superiorfrontal" -> "superior frontal gyrus"
print(len(DK_TO_ALLEN), "cortex parcels in the table, e.g.", list(DK_TO_ALLEN.items())[:2])

# leaf -> its Allen node (the leaf IS this structure; the name alone does not find it)
MANUAL = {
    "lateral_nucleus":                       10367,   # La, lateral nucleus of the amygdala
    "dorsal_raphe_nucleus":                  12223,   # DR, dorsal raphe nucleus (midbrain)
    "decussation_of_trochlear_nerve_fibres": 12338,   # x4, Allen spells it "fibers" (MWM)
    "reticular_formation_/_cuneiform_area":  12239,   # MRF, midbrain reticular formation (MGM)
}

# leaf -> Allen id of its father (no usable Allen node of its own; the leaf hangs from this one)
PLACE_UNDER = {
    "midbrain_(mesencephalon)":     10649,   # MGM; its own Allen node M is above the colors
    "superficial_pulvinar_nucleus": 10409,   # Pul, pulvinar of thalamus
    "diffuse_pulvinar_nucleus":     10409,   # Pul
    "parietotemporopontine_tract":  10668,   # HWM, white matter of hindbrain
    "corticonuclear_tract":         10668,   # HWM
    "frontopontine_tract":          10668,   # HWM
}

36 cortex parcels in the table, e.g. [('unknown', 'cerebral gyri and lobules'), ('bankssts', 'cerebral gyri and lobules')]


### 4.4 Match

For each leaf, `find_allen_node` returns `(allen_id, None)` when its Allen node is found (by
`MANUAL` or by name), and `(None, reason)` when it is not. Then:

- a leaf of `PLACE_UNDER` goes to `placed` (leaf → its own Allen node, or `None`), whatever the
  result: its father is already decided;
- a leaf whose Allen node is below a color goes to `match`;
- any other leaf goes to `unmatched_leaves` and is **not in the tree**. With the decisions of
  4.3 this list is empty. It is kept as a guard: a label added to the YAML in the future that
  cannot be matched shows up here instead of disappearing silently.

In [9]:
def find_allen_node(leaf):
    if leaf in MANUAL:
        return MANUAL[leaf], None
    name = leaf
    if leaf.startswith("ctx-rh-"):
        name = DK_TO_ALLEN.get(leaf[len("ctx-rh-"):], leaf)
    candidates = name_index.get(TA.normalize_name(name), [])
    if len(candidates) == 0:
        return None, "no Allen node with this name"
    if len(candidates) > 1:
        return None, f"several Allen nodes with this name: {candidates}"
    return candidates[0], None

match = {}        # leaf name -> allen id (placed by the rules of step 5)
placed = {}       # leaf name -> its own allen id, or None (father given in PLACE_UNDER)
unmatched = []    # (leaf name, labels, reason)
for leaf in labels_of:
    aid, reason = find_allen_node(leaf)
    if leaf in PLACE_UNDER:
        placed[leaf] = aid
    elif aid is None:
        unmatched.append((leaf, labels_of[leaf], reason))
    elif color_of(aid) is None:
        unmatched.append((leaf, labels_of[leaf], f"Allen node {allen[aid]['acronym']} is not below any color"))
    else:
        match[leaf] = aid

unmatched_leaves = pd.DataFrame(unmatched, columns=["leaf", "labels", "reason"])
print(f"{len(match)} leaves matched, {len(placed)} placed by hand, {len(unmatched_leaves)} not matched")
pd.DataFrame([(leaf, labels_of[leaf], allen[PLACE_UNDER[leaf]]["acronym"],
               allen[aid]["acronym"] if aid else None) for leaf, aid in placed.items()],
             columns=["placed leaf", "labels", "father", "own Allen node"])

327 leaves matched, 6 placed by hand, 0 not matched


,placed leaf,labels,father,own Allen node
0,midbrain_(mesencephalon),[384],MGM,M
1,superficial_pulvinar_nucleus,[811],Pul,NaN
2,diffuse_pulvinar_nucleus,[813],Pul,NaN
3,parietotemporopontine_tract,[827],HWM,NaN
4,corticonuclear_tract,[828],HWM,NaN
5,frontopontine_tract,[829],HWM,NaN


## 5. Which Allen nodes stay in the tree?

Call **X** the Allen node matched by a leaf. In the tree a node is either a **leaf** (it has
labels, no children) or **internal** (it has children, no labels). Two cases:

- **The leaf takes X's place** when nothing else needs X. The leaf hangs where X was, under X's
  Allen parent. Example: `anterodorsal_nucleus_of_thalamus` (219) → Allen `AD` has no other
  leaf below it, so the leaf replaces `AD` under its Allen parent `ANC` (anterior nuclear
  complex).
- **X stays as an internal node, and the leaf becomes its child**, when X is needed as a
  father: X is a color, or other leaves are below X, or several leaves match X. Example:
  `thalamus` (218) → Allen `THM`, but all the thalamic nuclei are also below `THM`. So `THM`
  stays, and the leaf `thalamus` (= "thalamus, not assigned to a nucleus") is one of its
  children, next to the nuclei. Same for `putamen` (79): Allen `Pu` also contains the leaf
  `posteroventral_putamen`, so `Pu` stays and `putamen` is its child.

The leaves of `PLACE_UNDER` (4.3, case 4) skip these rules: their father is the Allen node given
there, and they are always its child. They are kept apart in `placed_under` (father id →
leaves), not in `leaves_of`, so their own Allen node (e.g. `M`) plays no role in the placement
of the other leaves. Their father counts as "having a leaf below", so it always stays.

`keep` is the set of Allen nodes that stay as internal nodes: for every leaf, its father and
all the Allen nodes above the father, up to the color.

In [10]:
leaves_of = defaultdict(list)          # allen id -> leaves matched to it (placed by the rules)
for leaf, aid in match.items():
    leaves_of[aid].append(leaf)

placed_under = defaultdict(list)       # father allen id -> leaves placed by hand (PLACE_UNDER)
for leaf in placed:
    placed_under[PLACE_UNDER[leaf]].append(leaf)

has_leaf_below = {a for aid in leaves_of for a in ancestors(aid)} | set(placed_under)

def stays_internal(aid):
    return aid in color_ids or aid in has_leaf_below or len(leaves_of[aid]) > 1

def father_of(leaf):
    if leaf in PLACE_UNDER:
        return PLACE_UNDER[leaf]
    aid = match[leaf]
    return aid if stays_internal(aid) else allen[aid]["parent"]

keep = set()
for leaf in list(match) + list(placed):
    a = father_of(leaf)
    while True:                        # climb from the father to the color
        keep.add(a)
        if a in color_ids:
            break
        a = allen[a]["parent"]

kept_colors = [c for c in color_ids if c in keep]
print(len(keep), "Allen nodes kept as internal nodes")
print("colors with leaves:", [allen[c]["acronym"] for c in kept_colors])
print("colors without leaves (removed):", [allen[c]["acronym"] for c in color_ids if c not in keep])

161 Allen nodes kept as internal nodes
colors with leaves: ['FGM', 'FWM', 'FV', 'FSS', 'MGM', 'MWM', 'MSS', 'HGM', 'HWM', 'HSS']
colors without leaves (removed): ['fbv', 'MV', 'mbv', 'HV', 'hbv']


## 6. Build the tree with `AllenLabelTree`

### The class

The generator reads trees with `LabelTree` (`wp3/generator_v2_modular/nxgen/core/tree.py`).
`LabelTree` does not accept extra node fields, so `allen_tree.py` defines a subclass:

- `AllenNode(Node)`: a node with four more optional fields, `allen_id`, `allen_acronym`,
  `allen_name`, `allen_color`;
- `AllenLabelTree(LabelTree)`: a tree made of `AllenNode`s. Everything else is inherited from
  `LabelTree` (validation, visibility, label groups, `to_json`…), so it can be given to the
  generator functions like a `LabelTree`. `to_label_tree()` returns the plain `LabelTree` version.

Two methods build the tree:

- `AllenLabelTree.new(colors, ...)` creates a tree with only the `root` node (level 0);
- `tree.add_node(name, parent, **fields)` adds a node under an existing parent. It sets the
  `level` (parent level + 1) and adds the node to the parent's `children`. So we add the nodes
  **from the top down**.

### Node fields

| Field | Meaning | Value here |
|---|---|---|
| `labels` | labels painted with this node's intensity | leaves only |
| `w_mu` | level 1 only: weight of each color in the node's mean | one-hot: each color is independent |
| `wr_mu`, `wr_sigma` | random variation of the node's mean / sigma | same values as `nextbrain_tree_v1.json` |
| `r_max` | coarsest resolution (mm) at which the node is visible | `None` = always visible |
| `null` | intensity 0 (mean = sigma = 0) | background only |
| `allen_*` | the Allen node | internal node: itself; leaf: the node it matched; background: none |

### Node names

- leaves: the NextBrain name (`putamen`, `ctx-rh-superiorfrontal`), and `background`;
- internal nodes: Allen name in snake_case + `_l<level>`, like in `nextbrain_tree_v1.json`
  (`thalamus_l3`). If the name is already used, the Allen id is added.

In [11]:
LEVEL1 = dict(wr_mu=4.0, wr_sigma=0.5, r_max=None)
INTERNAL = dict(wr_mu=5.0, wr_sigma=0.5, r_max=None)
LEAF = dict(wr_mu=3.0, wr_sigma=0.5, r_max=None)

tree = AllenLabelTree.new(
    colors=[allen[c]["acronym"] for c in kept_colors],
    contrasts=["T1", "T2", "FLAIR", "PD"],
    version="nextbrain-allen-gyral-0.3",
    description=("NextBrain leaves (NextBrainLabels.yaml) on the Allen developmental ontology, "
                 "cortex on Allen's gyral hierarchy. Background (label 0) is a null leaf under "
                 "the root. Level 1 = Allen tissue classes of forebrain/midbrain/hindbrain. "
                 "Built in nextbrain_to_allen_tree.ipynb."),
)

def snake(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def allen_fields(aid):
    return dict(allen_id=aid, allen_acronym=allen[aid]["acronym"],
                allen_name=allen[aid]["name"], allen_color=allen[aid]["color"])

**The background** goes first, directly under the root. It is `null` (intensity 0), so it needs
no `w_mu` and it is not one of the `colors`.

In [12]:
tree.add_node("background", "root", labels=background_labels, null=True,
              wr_mu=0.0, wr_sigma=0.0, r_max=None,
              description="Background (NextBrainLabels.yaml: Unknown)")
print(tree.nodes["background"].to_dict())

{'parent': 'root', 'children': [], 'level': 1, 'labels': [0], 'wr_mu': 0.0, 'wr_sigma': 0.0, 'r_max': None, 'null': True, 'description': 'Background (NextBrainLabels.yaml: Unknown)'}


**The Allen nodes and the leaves.** `add_allen_node(aid, parent)` adds the Allen node `aid`,
then goes through its Allen children, in Allen order:

- a child in `keep` → added as an internal node (the same function, recursively);
- a child not in `keep` but matched by a leaf → the leaf takes its place;
- any other child → it has no leaf below it: skipped (removed from the tree).

Finally the leaves matched to `aid` itself ("X stays, the leaf is its child") are added, and
the leaves placed under `aid` by hand (`placed_under`, e.g. `midbrain_(mesencephalon)` under MGM,
the two pulvinar nuclei under `Pul`).

`add_leaf` writes the `allen_*` fields of the leaf's own Allen node (from `match` or `placed`);
a placed leaf without an Allen node of its own gets no `allen_*` fields. Placed leaves get a
`description` that records the decision.

In [13]:
own_allen_node = {**match, **placed}    # leaf -> its own Allen id (None: it has none)

def add_leaf(leaf, parent):
    aid = own_allen_node[leaf]
    description = ""
    if leaf in PLACE_UNDER:
        description = f"placed by hand under {allen[PLACE_UNDER[leaf]]['acronym']} (PLACE_UNDER)"
        if aid is None:
            description += "; no Allen node of its own"
    tree.add_node(leaf, parent, labels=labels_of[leaf], **LEAF, description=description,
                  **(allen_fields(aid) if aid is not None else {}))

def add_allen_node(aid, parent):
    level = tree.nodes[parent].level + 1
    name = f"{snake(allen[aid]['name'])}_l{level}"
    if name in tree.nodes:
        name = f"{snake(allen[aid]['name'])}_{aid}_l{level}"

    if level == 1:   # a color: its mean is 100% its own color
        tree.add_node(name, parent, w_mu=[float(c == aid) for c in kept_colors],
                      **LEVEL1, **allen_fields(aid))
    else:
        tree.add_node(name, parent, **INTERNAL, **allen_fields(aid))

    for child in allen[aid]["children"]:
        if child in keep:
            add_allen_node(child, name)
        else:
            for leaf in leaves_of[child]:    # empty if no leaf matched this child
                add_leaf(leaf, name)
    for leaf in leaves_of[aid] + placed_under[aid]:
        add_leaf(leaf, name)

for color in kept_colors:
    add_allen_node(color, "root")

n_leaves = len(tree.leaves())
print(f"{len(tree.nodes)} nodes: {n_leaves} leaves (with background), "
      f"{len(tree.nodes) - n_leaves - 1} internal nodes + root")
print("root children:", tree.nodes["root"].children)

496 nodes: 334 leaves (with background), 161 internal nodes + root
root children: ['background', 'gray_matter_of_forebrain_l1', 'white_matter_of_forebrain_l1', 'ventricles_of_forebrain_l1', 'surface_structures_of_forebrain_l1', 'gray_matter_of_midbrain_l1', 'white_matter_of_midbrain_l1', 'surface_structures_of_midbrain_l1', 'gray_matter_of_the_hindbrain_l1', 'white_matter_of_hindbrain_l1', 'surface_structures_of_hindbrain_l1']


## 7. Fathers without leaves

These are the Allen nodes below the 15 colors that are **not** in the tree: neither kept as an
internal node nor replaced by a leaf. They had no NextBrain leaf below them, so they were
removed. `is_color = True` marks the colors that were removed completely.

In [14]:
allen_ids_in_tree = {node.allen_id for node in tree.nodes.values()}

def subtree(aid):
    yield aid
    for child in allen[aid]["children"]:
        yield from subtree(child)

removed = [(aid, allen[aid]["acronym"], allen[aid]["name"], allen[color]["acronym"], aid == color)
           for color in color_ids for aid in subtree(color) if aid not in allen_ids_in_tree]
fathers_without_leaves = pd.DataFrame(removed, columns=["allen_id", "acronym", "name", "color", "is_color"])

print(len(fathers_without_leaves), "Allen nodes removed. Per color:")
print(fathers_without_leaves["color"].value_counts(sort=False).to_string())
fathers_without_leaves

2830 Allen nodes removed. Per color:
color
FGM    1857
FWM     117
FV       12
FSS     146
fbv       1
MGM     120
MWM      37
MV        2
MSS       7
mbv       1
HGM     377
HWM      67
HV        6
HSS      79
hbv       1


,allen_id,acronym,name,color,is_color
0,10506,FTS,transient structures of forebrain,FGM,False
1,11580,SG,subpial granular zone,FGM,False
2,11581,fSG,SG in frontal neocortex,FGM,False
3,11587,fSGfp,SG in frontal polar cortex,FGM,False
4,11582,fSGdm,SG in dorsomedial frontal cortex,FGM,False
...,...,...,...,...,...
2825,12886,r12,root of hypoglossal nerve,HSS,False
2826,12887,r9,root of glossopharyngeal nerve,HSS,False
2827,12888,r10,root of vagus nerve,HSS,False
2828,12889,vmms,ventral (anterior) median medullary sulcus,HSS,False


## 8. Checks

1. `tree.validate()` runs the generator's own checks: parents and children agree, levels are
   parent + 1, every color has `w_mu` with one weight per color, no label is used twice, every
   node is reachable from the root.
2. `tree.to_label_tree()` builds the plain generator `LabelTree` (without `allen_*`) and
   validates it again.
3. Every YAML entry is in exactly one place: a leaf of the tree, the background, or the list of
   unmatched leaves.
4. Leaves have labels and internal nodes do not: no father without leaves is left.
5. Every `PLACE_UNDER` leaf hangs directly from the Allen node given there, and keeps its own
   Allen node (or none).

In [15]:
tree.validate()
tree.to_label_tree()

in_tree = set(tree.leaves()) - {"background"}
assert in_tree.isdisjoint(unmatched_leaves["leaf"])
assert in_tree | set(unmatched_leaves["leaf"]) | {BACKGROUND} == set(leaves)

for name, node in tree.nodes.items():
    if name != "root":
        assert bool(node.labels) == (not node.children), name

for leaf, father_id in PLACE_UNDER.items():
    assert tree.nodes[tree.nodes[leaf].parent].allen_id == father_id, leaf
    assert tree.nodes[leaf].allen_id == placed[leaf], leaf

print("all checks OK")

all checks OK


## 9. Look at the tree and save it

Number of leaves per color, and the thalamus branch (leaves marked `*`, Allen acronym in
brackets). Note the leaf `thalamus` under `thalamus_l3`, next to the nuclei (step 5).

In [16]:
def count_leaves(name):
    return sum(1 for n in [name] + tree.descendants(name) if not tree.nodes[n].children)

def print_branch(name, depth=0):
    node = tree.nodes[name]
    if node.children:
        print("    " * depth + f"{name} [{node.allen_acronym}]")
        for child in node.children:
            print_branch(child, depth + 1)
    else:
        print("    " * depth + f"* {name} {node.labels} [{node.allen_acronym}]")

print({c: count_leaves(c) for c in tree.nodes["root"].children})
print()
print("children of MGM:", tree.nodes["gray_matter_of_midbrain_l1"].children)
print("children of HWM:", tree.nodes["white_matter_of_hindbrain_l1"].children)
print("children of Pul:", tree.nodes["pulvinar_of_thalamus_l7"].children)
print()
print_branch("thalamus_l3")

{'background': 1, 'gray_matter_of_forebrain_l1': 179, 'white_matter_of_forebrain_l1': 20, 'ventricles_of_forebrain_l1': 1, 'surface_structures_of_forebrain_l1': 36, 'gray_matter_of_midbrain_l1': 23, 'white_matter_of_midbrain_l1': 10, 'surface_structures_of_midbrain_l1': 2, 'gray_matter_of_the_hindbrain_l1': 43, 'white_matter_of_hindbrain_l1': 17, 'surface_structures_of_hindbrain_l1': 2}

children of MGM: ['pretectal_region', 'midbrain_tegmentum_l2', 'midbrain_tectum_l2', 'midbrain_(mesencephalon)']
children of HWM: ['amiculum_of_the_olive', 'central_tegmental_tract', 'inferior_cerebellar_peduncle', 'lateral_lemniscus', 'medial_lemniscus', 'medial_longitudinal_fasciculus', 'mesencephalic_trigeminal_tract', 'middle_cerebellar_peduncle_l2', 'pyramidal_tract_l2', 'solitary_tract', 'spinal_lemniscus_in_hindbrain', 'spinal_trigeminal_tract', 'white_matter_of_hindbrain', 'parietotemporopontine_tract', 'corticonuclear_tract', 'frontopontine_tract']
children of Pul: ['anterior_nucleus_of_pulvin

Four leaves and one internal node, as they are written in the JSON file. Note the two placed
leaves (4.3, case 4): `midbrain_(mesencephalon)` hangs from MGM but its `allen_*` fields are
those of `M`, the node its name matches; `superficial_pulvinar_nucleus` hangs from `Pul` and has
no `allen_*` fields, because Allen has no node for it.

In [17]:
for name in ["anterodorsal_nucleus_of_thalamus", "putamen", "midbrain_(mesencephalon)",
             "superficial_pulvinar_nucleus", "thalamus_l3"]:
    print(name, tree.nodes[name].to_dict())

anterodorsal_nucleus_of_thalamus {'parent': 'anterior_nuclear_complex_of_thalamus_l5', 'children': [], 'level': 6, 'labels': [219], 'wr_mu': 3.0, 'wr_sigma': 0.5, 'r_max': None, 'allen_id': 10393, 'allen_acronym': 'AD', 'allen_name': 'anterodorsal nucleus of thalamus', 'allen_color': 'B67BB4'}
putamen {'parent': 'putamen_l6', 'children': [], 'level': 7, 'labels': [79], 'wr_mu': 3.0, 'wr_sigma': 0.5, 'r_max': None, 'allen_id': 10338, 'allen_acronym': 'Pu', 'allen_name': 'putamen', 'allen_color': 'A8B485'}
midbrain_(mesencephalon) {'parent': 'gray_matter_of_midbrain_l1', 'children': [], 'level': 2, 'labels': [384], 'wr_mu': 3.0, 'wr_sigma': 0.5, 'r_max': None, 'description': 'placed by hand under MGM (PLACE_UNDER)', 'allen_id': 10648, 'allen_acronym': 'M', 'allen_name': 'midbrain (mesencephalon)', 'allen_color': 'AFD0DD'}
superficial_pulvinar_nucleus {'parent': 'pulvinar_of_thalamus_l7', 'children': [], 'level': 8, 'labels': [811], 'wr_mu': 3.0, 'wr_sigma': 0.5, 'r_max': None, 'descripti

### Save

`tree.to_json` writes the tree with all its fields, including `allen_*` and the background.
Read it back with `AllenLabelTree.from_json(path)` (the plain `LabelTree.from_json` does not
accept the `allen_*` fields; use `AllenLabelTree.from_json(path)` and pass the tree, or its
`.to_label_tree()`, to the generator).

In [18]:
tree.to_json(OUT_TREE)
assert AllenLabelTree.from_json(OUT_TREE).to_dict() == tree.to_dict()   # reloads identically
print("saved", OUT_TREE.resolve())

saved /home/agustin/phd/synthseg_nextbrain/project/dev/wp1/allen_ontology/data/outputs/nextbrain_tree_allen_gyral.json


## 10. Final check: are all NextBrain labels in the tree?

An independent check on the **saved file**. It does not use any variable from the steps above:

1. read every label of `NextBrainLabels.yaml` (an entry can have several labels, e.g.
   `white_matter_of_hindbrain: [846, 611]`);
2. read the saved tree and collect the labels painted by its nodes (`label_to_node`);
3. a label that is in the YAML but not in the tree is **missing**.

The expected answer is *no missing labels*.

In [19]:
# 1. every NextBrain label -> its name
nextbrain = {}
for name, labels in io.load_yaml(LUTDIR / "NextBrainLabels.yaml").items():
    for label in (labels if isinstance(labels, list) else [labels]):
        nextbrain[label] = name

# 2. every label of the saved tree -> the node that paints it
saved_tree = AllenLabelTree.from_json(OUT_TREE)
in_tree = saved_tree.label_to_node

# 3. labels in NextBrain but not in the tree
missing = sorted(set(nextbrain) - set(in_tree))

print(f"NextBrain labels: {len(nextbrain)}   labels in the tree: {len(in_tree)}   missing: {len(missing)}")
for label in missing:
    print(f"   {label}: {nextbrain[label]}")

NextBrain labels: 335   labels in the tree: 335   missing: 0


## 11. Resolution limit `r_max` of every node

`r_max` is the coarsest voxel size (mm) at which the generator still draws a node; at coarser
resolutions the node is merged into its parent (a node is visible iff `resolution <= r_max` and its
parent is visible).

Here `r_max` = `min_resolution_mm` of `data/outputs/node_thickness.csv`, the table written by
`node_thickness.py`: the typical thickness of the node in the NextBrain templates divided by 2, so
that the structure is at least 2 voxels across (see the docstring of that script). The same value
is used for every contrast.

- Nodes without a measurement (empty `min_resolution_mm`: the lobes and gyri, absent from the
  templates because there the whole cortex is a single label), the root and the background keep
  `r_max = null`, so they are visible whenever their parent is.
- `r_max` on level-1 nodes is ignored by the generator (they are always visible); `validate()`
  warns about it.
- A child can get a larger `r_max` than its parent (it is thicker than the parent's typical
  thickness). It is never seen beyond the parent limit anyway, so the generated images are the same
  with both options of `CAP_TO_PARENT`:
  - `CAP_TO_PARENT = True`: the child's `r_max` is lowered to its parent's (going down the tree,
    so a cap propagates to the grandchildren). The stored values are then consistent and
    `validate()` does not warn. Children of level-1 nodes are not capped (level-1 `r_max` is
    ignored). Nodes without a measurement under a measured parent (the lobes and gyri) get the
    parent's value, which is what null already meant.
  - `CAP_TO_PARENT = False`: the raw csv values are kept and `validate()` warns once per such child.

The tree is read from `OUT_TREE`, its `r_max` are reset and set again, and it is saved back to
`OUT_TREE`.

In [20]:
import warnings

THICKNESS_CSV = Path("data/outputs/node_thickness.csv")   # written by node_thickness.py
CAP_TO_PARENT = True   # True: r_max of a child <= r_max of its parent; False: raw csv values

# 1. node -> min_resolution_mm (nodes with no measurement are left out)
thickness = pd.read_csv(THICKNESS_CSV)
r_max = thickness.dropna(subset=["min_resolution_mm"]).set_index("node")["min_resolution_mm"]

# warnings of validate() (raised when a tree is built or read) are collected and summarised below
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    # 2. set r_max on the saved tree (same value for every contrast); every node is reset to null
    #    first, so re-running the cell with either CAP_TO_PARENT gives the same tree
    rmax_tree = AllenLabelTree.from_json(OUT_TREE)
    unknown = sorted(set(r_max.index) - set(rmax_tree.nodes))
    assert not unknown, f"nodes of the csv that are not in the tree: {unknown}"
    for name in rmax_tree.nodes:
        rmax_tree.nodes[name].r_max = {"default": float("inf")}   # same as r_max: null in the json
    for name, r in r_max.items():
        rmax_tree.nodes[name].r_max = {"default": float(r)}       # same as r_max: <float> in the json

    # 3. optional cap, parents before children (tree order). Level-1 parents are skipped: their
    #    r_max is ignored by the generator (always visible), as in validate()
    capped = []
    if CAP_TO_PARENT:
        for name in rmax_tree.order():
            node = rmax_tree.nodes[name]
            if node.level is None or node.level < 2 or rmax_tree.nodes[node.parent].level < 2:
                continue
            limit = rmax_tree.nodes[node.parent].r_max["default"]
            if node.r_max["default"] > limit:
                capped.append(name)
                node.r_max = {"default": limit}

    # 4. validate and save
    caught.clear()
    rmax_tree.validate()
    rmax_tree.to_json(OUT_TREE)
    assert AllenLabelTree.from_json(OUT_TREE).to_dict() == rmax_tree.to_dict()   # reloads identically

messages = sorted({str(w.message) for w in caught})
no_rmax = [n for n in rmax_tree.order() if rmax_tree.nodes[n].r_max["default"] == float("inf")]
print(f"r_max from the csv on {len(r_max)} nodes; CAP_TO_PARENT={CAP_TO_PARENT}: {len(capped)} capped")
if capped:
    print("   capped: " + ", ".join(capped))
print(f"left null on {len(no_rmax)}: " + ", ".join(no_rmax))
print(f"validate: {len(messages)} warnings")
for kind in ("level-1", "parent r_max"):
    ws = [m for m in messages if kind in m]
    print(f"   {len(ws)} x '{kind}'" + (f", e.g. {ws[0]}" if ws else ""))
print("saved", OUT_TREE.resolve())

r_max from the csv on 453 nodes; CAP_TO_PARENT=True: 109 capped
   capped: cerebral_nuclei_l3, thalamus_l3, anterior_commissure, fimbria, fornix, optic_chiasm, optic_tract, frontal_lobe_l3, parietal_lobe_l3, temporal_lobe_l3, occipital_lobe_l3, ctx-rh-insula, ctx-rh-bankssts, red_nucleus, superior_cerebellar_peduncle_(brachium_conjunctivum), cerebellum_l3, middle_cerebellar_peduncle, rostral_(anterior)_medullary_velum, basal_nuclei_basal_ganglia_l4, ctx-rh-precentral, ctx-rh-superiorfrontal, middle_frontal_gyrus_l4, inferior_frontal_gyrus_l4, ctx-rh-medialorbitofrontal, ctx-rh-lateralorbitofrontal, ctx-rh-paracentral, ctx-rh-frontalpole, ctx-rh-postcentral, ctx-rh-superiorparietal, inferior_parietal_lobule_l4, ctx-rh-precuneus, ctx-rh-superiortemporal, ctx-rh-middletemporal, ctx-rh-inferiortemporal, ctx-rh-transversetemporal, ctx-rh-temporalpole, ctx-rh-cuneus, ctx-rh-lingual, ctx-rh-fusiform, ctx-rh-lateraloccipital, ctx-rh-pericalcarine, parahippocampal_gyrus_l4, peripeduncular_nucle